# Cohort Extraction

Extracts Sepsis-3 patients- adult ICU admissions with infection and SOFA ≥ 2 within the first 24h.

Output: `data/cohort/sepsis3_cohort.csv`

In [1]:
import os
import configparser
import psycopg2
import sqlalchemy
import pandas as pd
import amsterdamumcdb as adb

pd.set_option('display.max_columns', None)

## Connect to database

In [ ]:
config = configparser.ConfigParser()
config.read(os.path.join('..', 'AmsterdamUMCdb', 'config.ini'))

output_dir = os.path.join('..', 'data', 'cohort')
os.makedirs(output_dir, exist_ok=True)

# Two connections needed: psycopg2 for session settings, SQLAlchemy for the amsterdamumcdb package
# Claude AI. prompt- How do I connect to the db using psycog2 and sqlalchemy?
pg_con = psycopg2.connect(
    database=config['psycopg2']['database'],
    user=config['psycopg2']['username'],
    password=config['psycopg2']['password'],
    host=config['psycopg2']['host'],
    port=config['psycopg2']['port']
)
pg_con.set_client_encoding('WIN1252')
pg_con.set_session(autocommit=True)

engine = sqlalchemy.create_engine(
    f"postgresql+psycopg2://{config['psycopg2']['username']}:{config['psycopg2']['password']}"
    f"@{config['psycopg2']['host']}:{config['psycopg2']['port']}/{config['psycopg2']['database']}"
    f"?options=-csearch_path%3Dpublic%20-cclient_encoding%3DWIN1252"
)
con = engine.connect()
print('Connected')

Connected


## Load admissions and apply inclusion filters


In [ ]:
admissions = pd.read_sql("SELECT * FROM admissions", con)
print(f'Total admissions: {admissions.admissionid.nunique()}')
admissions[['admissionid', 'patientid', 'location', 'agegroup', 'gender', 'weightgroup']].head()

In [ ]:
# ICU only (excludes medium care units)
# Claude AI. prompt- For the admissions datatype, how do we include only ICU and adults? 
icu = admissions[admissions['location'].str.contains('IC', na=False)]
print(f'ICU admissions: {icu.admissionid.nunique()}')
print(admissions['location'].value_counts())

In [ ]:
# Adults only - agegroup field is populated for all adult patients
adults = icu[icu['agegroup'].notna() & ~icu['agegroup'].isin(['None', ''])]
print(f'Adult ICU admissions: {adults.admissionid.nunique()}')
print(adults['agegroup'].value_counts().sort_index())

## Extract Sepsis-3 cohort



In [ ]:
sepsis = adb.get_sepsis_patients(con)
print(f'Sepsis-3 (all locations): {sepsis.admissionid.nunique()}')

In [ ]:
adult_icu_ids = set(adults['admissionid'])
sepsis = sepsis[sepsis['admissionid'].isin(adult_icu_ids)]
print(f'Sepsis-3 adult ICU: {sepsis.admissionid.nunique()}')

print(f'\nSOFA score distribution:')
print(sepsis['sofa_total_score'].describe())

print(f'\nInfection criteria:')
print(f'  Physician-flagged:         {(sepsis["sepsis_at_admission"] == 1).sum()}')
print(f'  Therapeutic antibiotics:   {(sepsis["sepsis_antibiotics_bool"] == 1).sum()}')
print(f'  Antibiotics + cultures:    {((sepsis["other_antibiotics_bool"] == 1) & (sepsis["sepsis_cultures_bool"] == 1)).sum()}')

## Merge metadata and save

In [ ]:
# ClaudeAI. Prompt- Need to merge two dataframes. Only common rows of dataframe need to be merged. 
cols = ['admissionid', 'patientid', 'admittedat', 'dischargedat',
        'location', 'urgency', 'agegroup', 'gender', 'weightgroup', 'lengthgroup']
cols = [c for c in cols if c in admissions.columns]

sepsis = pd.merge(sepsis, admissions[cols], on='admissionid', how='left', suffixes=('', '_adm'))

output_path = os.path.join(output_dir, 'sepsis3_cohort.csv')
sepsis.to_csv(output_path, index=False)
print(f'Saved {sepsis.admissionid.nunique()} admissions to {output_path}')

pg_con.close()